In [4]:
import anndata as ad
import h5py
import numpy as np
import pandas as pd
import scipy.sparse as sp
from pathlib import Path
from configs import HEAVY, ROOT, TABLES


In [11]:
SOURCES = {"ts": Path("D:/tabula/raw"), "brain": Path("D:/tabula/raw/brain")}
INVENTORY = ["ts", "brain"]
MAX_LEVELS = 400
SAMPLE_ROWS = 200
KEYWORDS = ["cell_type", "tissue", "donor", "compartment", "ontology", "free_annotation", "roi", "region",
            "dissection", "supercluster", "class", "cluster", "neuron"]

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

In [8]:
def inventory(tag, folder):
    files = sorted(folder.glob("*.h5ad"))
    print(f"\n######## {tag}: {len(files)} h5ad files in {folder}")
    summary, counts = [], []
    for f in files:
        a = ad.read_h5ad(f, backed="r")
        x = a.X[:SAMPLE_ROWS]
        x = x.toarray() if hasattr(x, "toarray") else np.asarray(x)
        nz = x[x != 0]
        summary.append({
            "file": f.name, "size_gb": round(f.stat().st_size / 1024**3, 2), "cells": a.n_obs, "genes": a.n_vars,
            "X_dtype": str(x.dtype), "X_sample_integer": bool(np.all(np.mod(nz, 1) == 0)) if nz.size else None,
            "X_sample_max": float(nz.max()) if nz.size else 0.0,
            "layers": ",".join(map(str, a.layers.keys())), "has_raw": a.raw is not None,
            "var_index_example": ",".join(map(str, a.var_names[:3])), "var_columns": ",".join(map(str, a.var.columns)),
        })
        columns = []
        for col in a.obs.columns:
            s = a.obs[col]
            n = s.nunique()
            columns.append({"column": col, "dtype": str(s.dtype), "levels": n,
                            "examples": "; ".join(map(str, s.value_counts().index[:5]))})
            if n <= MAX_LEVELS and any(k in col.lower() for k in KEYWORDS):
                vc = s.value_counts()
                counts += [{"file": f.name, "column": col, "value": v, "cells": int(c)} for v, c in vc.items()]
        a.file.close()
        print(f"\n== {f.name}: {summary[-1]['cells']} cells, {summary[-1]['genes']} genes; obs columns")
        print(pd.DataFrame(columns).to_string(index=False))
    summary = pd.DataFrame(summary)
    print(f"\n== {tag} files")
    print(summary.to_string(index=False))
    summary.to_csv(TABLES / f"08_inventory_files_{tag}.csv", index=False)
    pd.DataFrame(counts).to_csv(TABLES / f"08_inventory_levels_{tag}.csv", index=False)


In [12]:
for tag in INVENTORY:
    inventory(tag, SOURCES[tag])


######## ts: 28 h5ad files in D:\tabula\raw

== bladder.h5ad: 66385 cells, 61806 genes; obs columns
                     column    dtype  levels                                                                                                                                                                                        examples
                      donor category       6                                                                                                                                                                 TSP21; TSP1; TSP2; TSP27; TSP25
                     tissue category       1                                                                                                                                                                                         Bladder
        anatomical_position category       1                                                                                                                                                

In [13]:
BUILD_PROFILES = True
TS_LAYER = "raw_counts"
MIN_CELLS = 50
CHUNK = 10000

In [14]:
CORTEX_FILES = ["cerebral cortex A1C", "cerebral cortex M1C", "cerebral cortex MTG", "cerebral cortex S1C",
                "cerebral cortex STG", "cerebral cortex V1C"]
HIPPO_REGION_FILE = "Hippocampal CA4-DGC"  
HIPPO_NEURON_FILES = ["Hippocampal CA1-3", "Hippocampal CA4-DG", "Hippocampal DG"] 
HIPPO_REGION_ROI = "Human CA4C-DGC" 
CEREBELLUM_CORTEX_FILES = ["Cerebellum CBL", "Cerebellum CBV"]
GLIA = ["astrocyte", "oligodendrocyte", "oligodendrocyte precursor cell", "central nervous system macrophage"]
GRANULAR_CLUSTERS = ["310", "311", "312", "308", "302"] 
MOLECULAR_CLUSTERS = ["303", "304", "305", "306", "307"]

In [15]:
def cell_types(o, types):
    return o["cell_type"].astype(str).isin(types).to_numpy()


def clusters(o, ids):
    return o["cluster_id"].astype(str).isin(ids).to_numpy()


def hippocampal_neurons_elsewhere(o):
    return ((o["cell_type"].astype(str) == "neuron") & o["dissection"].astype(str).str.contains("hippocampus", case=False)
            & (o["roi"].astype(str) != HIPPO_REGION_ROI)).to_numpy()

In [16]:
BRAIN = []
for f in CORTEX_FILES:
    BRAIN += [("Cerebral cortex | neuronal cells", f, lambda o: cell_types(o, ["neuron"])),
              ("Cerebral cortex | glial cells", f, lambda o: cell_types(o, GLIA)),
              ("Cerebral cortex | endothelial cells", f, lambda o: cell_types(o, ["endothelial cell"]))]
BRAIN += [("Hippocampus | neuronal cells", HIPPO_REGION_FILE, lambda o: cell_types(o, ["neuron"])),
          ("Hippocampus | glial cells", HIPPO_REGION_FILE, lambda o: cell_types(o, GLIA))]
BRAIN += [("Hippocampus | neuronal cells", f, hippocampal_neurons_elsewhere) for f in HIPPO_NEURON_FILES]
BRAIN += [("Caudate | neuronal cells", "Caudate", lambda o: cell_types(o, ["neuron"])),
          ("Caudate | glial cells", "Caudate", lambda o: cell_types(o, GLIA))]
for f in CEREBELLUM_CORTEX_FILES:
    BRAIN += [("Cerebellum | cells in granular layer", f, lambda o: clusters(o, GRANULAR_CLUSTERS)),
              ("Cerebellum | cells in molecular layer", f, lambda o: clusters(o, MOLECULAR_CLUSTERS))]

In [17]:
BRAIN

[('Cerebral cortex | neuronal cells',
  'cerebral cortex A1C',
  <function __main__.<lambda>(o)>),
 ('Cerebral cortex | glial cells',
  'cerebral cortex A1C',
  <function __main__.<lambda>(o)>),
 ('Cerebral cortex | endothelial cells',
  'cerebral cortex A1C',
  <function __main__.<lambda>(o)>),
 ('Cerebral cortex | neuronal cells',
  'cerebral cortex M1C',
  <function __main__.<lambda>(o)>),
 ('Cerebral cortex | glial cells',
  'cerebral cortex M1C',
  <function __main__.<lambda>(o)>),
 ('Cerebral cortex | endothelial cells',
  'cerebral cortex M1C',
  <function __main__.<lambda>(o)>),
 ('Cerebral cortex | neuronal cells',
  'cerebral cortex MTG',
  <function __main__.<lambda>(o)>),
 ('Cerebral cortex | glial cells',
  'cerebral cortex MTG',
  <function __main__.<lambda>(o)>),
 ('Cerebral cortex | endothelial cells',
  'cerebral cortex MTG',
  <function __main__.<lambda>(o)>),
 ('Cerebral cortex | neuronal cells',
  'cerebral cortex S1C',
  <function __main__.<lambda>(o)>),
 ('Cerebra

In [18]:
def group_sums(path, key, codes, n_groups):
    with h5py.File(path, "r") as fh:
        node = fh[key]
        if isinstance(node, h5py.Dataset):
            n_rows, n_cols = node.shape
            out = np.zeros((n_groups, n_cols))
            for s in range(0, n_rows, CHUNK):
                c = codes[s:s + CHUNK]
                keep = np.flatnonzero(c >= 0)
                if len(keep):
                    ind = sp.csr_matrix((np.ones(len(keep)), (c[keep], keep)), shape=(n_groups, len(c)))
                    out += ind @ node[s:s + CHUNK].astype(np.float64)
            return out
        attrs = {k: (v.decode() if isinstance(v, bytes) else v) for k, v in node.attrs.items()}
        enc = attrs.get("encoding-type", attrs.get("h5sparse_format", "") + "_matrix")
        n_rows, n_cols = (int(v) for v in attrs.get("shape", attrs.get("h5sparse_shape")))
        indptr = node["indptr"][:]
        out = np.zeros((n_groups, n_cols))
        if enc == "csr_matrix":
            for s in range(0, n_rows, CHUNK):
                e = min(s + CHUNK, n_rows)
                keep = np.flatnonzero(codes[s:e] >= 0)
                if not len(keep):
                    continue
                a, b = indptr[s], indptr[e]
                block = sp.csr_matrix((node["data"][a:b].astype(np.float64), node["indices"][a:b], indptr[s:e + 1] - a),
                                      shape=(e - s, n_cols))
                ind = sp.csr_matrix((np.ones(len(keep)), (codes[s:e][keep], keep)), shape=(n_groups, e - s))
                out += (ind @ block).toarray()
        elif enc == "csc_matrix":
            for s in range(0, n_cols, CHUNK):
                e = min(s + CHUNK, n_cols)
                a, b = indptr[s], indptr[e]
                rows = node["indices"][a:b]
                col = np.repeat(np.arange(s, e), np.diff(indptr[s:e + 1]))
                c = codes[rows]
                keep = c >= 0
                out += np.bincount(c[keep] * n_cols + col[keep], weights=node["data"][a:b][keep],
                                   minlength=n_groups * n_cols).reshape(n_groups, n_cols)
        else:
            raise ValueError(f"{path} {key}: unknown encoding {enc}")
        return out


def sum_units(path, key, units, gene_ids, keep_genes):
    codes = np.full(len(next(iter(units.values()))), -1)
    for i, m in enumerate(units.values()):
        assert (codes[m] == -1).all(), f"{path}: overlapping cell selections"
        codes[m] = i
    sums = group_sums(path, key, codes, len(units))
    df = pd.DataFrame(sums.T[keep_genes], index=gene_ids[keep_genes], columns=list(units))
    return df.groupby(level=0).sum()

In [19]:
counts, meta = {}, []

In [20]:
def add(pid, col, n_cells, donors, source, file, cells):
        counts[pid] = counts[pid].add(col, fill_value=0) if pid in counts else col
        meta.append({"profile_id": pid, "source": source, "file": file, "cells": cells, "n_cells": n_cells, "donors": set(donors)})

In [21]:
draft = pd.read_csv("../external/context_bridge_draft.csv")
draft["context"] = draft["hpa_tissue"] + " | " + draft["hpa_cell_type"]

for ctx in ["Skin | keratinocytes", "Skin | epidermal cells"]:
    draft.loc[draft["context"] == ctx, ["ts_file", "ts_cell_types", "status", "note"]] = [
        "tongue.h5ad", "stratified squamous epithelial cell", "proxy", "no epidermis in the skin file"]
ts = draft[draft["status"].isin(["match", "proxy"])].copy()
ts["types"] = ts["ts_cell_types"].str.split("; ").map(tuple)
ts["profile_id"] = "ts:" + ts["ts_file"].str.removesuffix(".h5ad") + ":" + ts["ts_cell_types"]

In [22]:
for fname, rows in ts.groupby("ts_file"):
    path = SOURCES["ts"] / fname
    a = ad.read_h5ad(path, backed="r")
    obs, var = a.obs, a.var
    a.file.close()
    with h5py.File(path, "r") as fh:
        assert f"layers/{TS_LAYER}" in fh, f"{fname}: no layer {TS_LAYER}"
    is10x = (obs["method"].astype(str) == "10X").to_numpy()
    cls = obs["cell_ontology_class"].astype(str).to_numpy()
    wanted = sorted({t for types in rows["types"] for t in types})
    missing = [t for t in wanted if not (cls == t).any()]
    assert not missing, f"{fname}: {missing} not in cell_ontology_class"
    units = {t: is10x & (cls == t) for t in wanted}
    gene_ids = var["ensembl_id"].astype(str).str.split(".").str[0].to_numpy()
    keep_genes = ~var["ercc"].astype(bool).to_numpy()
    sums = sum_units(path, f"layers/{TS_LAYER}", units, gene_ids, keep_genes)
    donors = obs["donor"].astype(str).to_numpy()
    for pid, types in rows.drop_duplicates("profile_id")[["profile_id", "types"]].itertuples(index=False):
        m = np.logical_or.reduce([units[t] for t in types])
        add(pid, sums[list(types)].sum(axis=1), int(m.sum()), donors[m], "ts", fname, "; ".join(types))
    print(f"{fname}: {len(rows.drop_duplicates('profile_id'))} profiles, 10X cells {int(is10x.sum())} of {len(is10x)}")

bladder.h5ad: 1 profiles, 10X cells 64214 of 66385
bone_marrow.h5ad: 1 profiles, 10X cells 23981 of 27112
fat.h5ad: 1 profiles, 10X cells 93195 of 94415
heart.h5ad: 1 profiles, 10X cells 25419 of 25832
kidney.h5ad: 1 profiles, 10X cells 11024 of 11376
l.intestine.h5ad: 3 profiles, 10X cells 28370 of 30084
liver.h5ad: 2 profiles, 10X cells 21398 of 22214
lung.h5ad: 2 profiles, 10X cells 62954 of 65847
lymph_node.h5ad: 3 profiles, 10X cells 126119 of 129062
mammary.h5ad: 2 profiles, 10X cells 30547 of 30936
muscle.h5ad: 1 profiles, 10X cells 41876 of 46772
ovary.h5ad: 2 profiles, 10X cells 45384 of 48951
pancreas.h5ad: 2 profiles, 10X cells 14140 of 14140
prostate.h5ad: 1 profiles, 10X cells 20405 of 21030
s.intestine.h5ad: 2 profiles, 10X cells 40350 of 42036
salivary_gland.h5ad: 1 profiles, 10X cells 38216 of 39821
skin.h5ad: 3 profiles, 10X cells 15757 of 17786
spleen.h5ad: 2 profiles, 10X cells 67601 of 70448
stomach.h5ad: 1 profiles, 10X cells 32690 of 33064
testis.h5ad: 2 profiles,

In [23]:
brain_files = sorted({f for _, f, _ in BRAIN})
for f in brain_files:
    path = SOURCES["brain"] / f"{f}.h5ad"
    a = ad.read_h5ad(path, backed="r")
    obs, var = a.obs, a.var
    a.file.close()
    rules = [(ctx, rule) for ctx, ff, rule in BRAIN if ff == f]
    units = {ctx: rule(obs) for ctx, rule in rules}
    gene_ids = pd.Index(var.index).astype(str).str.split(".").str[0].to_numpy()
    sums = sum_units(path, "X", units, gene_ids, np.ones(len(gene_ids), bool))
    donors = obs["donor_id"].astype(str).to_numpy()
    for ctx, m in units.items():
        add("brain:" + ctx, sums[ctx], int(m.sum()), donors[m], "brain", f, ctx)
    print(f"{f}.h5ad: " + ", ".join(f"{ctx.split(' | ')[1]} {int(m.sum())}" for ctx, m in units.items()))

Caudate.h5ad: neuronal cells 29889, glial cells 2637
Cerebellum CBL.h5ad: cells in granular layer 23868, cells in molecular layer 903
Cerebellum CBV.h5ad: cells in granular layer 45155, cells in molecular layer 9446
Hippocampal CA1-3.h5ad: neuronal cells 68452
Hippocampal CA4-DG.h5ad: neuronal cells 4594
Hippocampal CA4-DGC.h5ad: neuronal cells 36588, glial cells 19314
Hippocampal DG.h5ad: neuronal cells 47750
cerebral cortex A1C.h5ad: neuronal cells 30175, glial cells 2543, endothelial cells 19
cerebral cortex M1C.h5ad: neuronal cells 110246, glial cells 6172, endothelial cells 55
cerebral cortex MTG.h5ad: neuronal cells 85093, glial cells 21805, endothelial cells 131
cerebral cortex S1C.h5ad: neuronal cells 35453, glial cells 2689, endothelial cells 16
cerebral cortex STG.h5ad: neuronal cells 27340, glial cells 13084, endothelial cells 26
cerebral cortex V1C.h5ad: neuronal cells 30599, glial cells 2832, endothelial cells 15


In [24]:
meta = pd.DataFrame(meta).groupby("profile_id").agg(
    source=("source", "first"), files=("file", lambda x: "; ".join(sorted(set(x)))), cells=("cells", "first"),
    n_cells=("n_cells", "sum"), donors=("donors", lambda x: len(set().union(*x)))).reset_index()
raw = pd.DataFrame(counts)
meta["library_size"] = meta["profile_id"].map(raw.sum(axis=0))  # over each atlas's own genes, before intersecting
meta["kept"] = meta["n_cells"] >= MIN_CELLS

In [26]:
ts_genes = raw.loc[:, meta.loc[meta["source"] == "ts", "profile_id"]].notna().all(axis=1)
brain_genes = raw.loc[:, meta.loc[meta["source"] == "brain", "profile_id"]].notna().all(axis=1)
shared = raw.index[ts_genes & brain_genes]
kept = meta.loc[meta["kept"], "profile_id"].tolist()
profiles = np.log1p(raw.loc[shared, kept] / meta.set_index("profile_id").loc[kept, "library_size"] * 1e6)
print(f"\ngenes: Tabula Sapiens {int(ts_genes.sum())}, brain {int(brain_genes.sum())}, both {len(shared)}")


genes: Tabula Sapiens 61806, brain 58232, both 58182


In [27]:
contexts = pd.read_parquet(HEAVY / "01_labels.parquet", columns=["context", "tissue", "cell_type"]).drop_duplicates("context")
brain_ctx = pd.DataFrame({"context": sorted({c for c, _, _ in BRAIN})})
brain_ctx["profile_id"], brain_ctx["status"] = "brain:" + brain_ctx["context"], "match"
bridge = contexts.merge(pd.concat([ts[["context", "profile_id", "status", "note"]], brain_ctx]), on="context", how="left")
bridge = bridge.merge(meta[["profile_id", "n_cells", "donors"]], on="profile_id", how="left")
bridge.loc[bridge["profile_id"].isin(meta.loc[~meta["kept"], "profile_id"]), "status"] = "too few cells"
bridge["status"] = bridge["status"].fillna("none")
bridge.loc[bridge["status"].isin(["none", "too few cells"]), "profile_id"] = None
bridge["profile_shared_by"] = bridge.groupby("profile_id")["context"].transform("size")

In [28]:
bridge

,context,tissue,cell_type,profile_id,status,note,n_cells,donors,profile_shared_by
0,Adipose tissue | adipocytes,Adipose tissue,adipocytes,NaN,none,NaN,NaN,NaN,NaN
1,Adrenal gland | glandular cells,Adrenal gland,glandular cells,NaN,none,NaN,NaN,NaN,NaN
2,Appendix | glandular cells,Appendix,glandular cells,ts:l.intestine:enterocyte of epithelium of large intestine; large intestine ...,proxy,same profile as Colon,5225.0,5.0,3.0
3,Appendix | lymphoid tissue,Appendix,lymphoid tissue,"ts:lymph_node:b cell; cd4-positive, alpha-beta t cell; cd8-positive, alpha-b...",proxy,no appendix file,108486.0,6.0,1.0
4,Bone marrow | hematopoietic cells,Bone marrow,hematopoietic cells,ts:bone_marrow:hematopoietic stem cell; hematopoietic precursor cell; common...,match,NaN,3488.0,6.0,1.0
...,...,...,...,...,...,...,...,...,...
70,Tonsil | squamous epithelial cells,Tonsil,squamous epithelial cells,ts:tongue:stratified squamous epithelial cell,proxy,no tonsil file,12251.0,4.0,7.0
71,Urinary bladder | urothelial cells,Urinary bladder,urothelial cells,ts:bladder:bladder urothelial cell,match,NaN,26663.0,6.0,1.0
72,Vagina | squamous epithelial cells,Vagina,squamous epithelial cells,ts:tongue:stratified squamous epithelial cell,proxy,no vagina file,12251.0,4.0,7.0
73,Ovary | follicle cells,Ovary,follicle cells,ts:ovary:granulosa cell; theca cell,match,NaN,1679.0,3.0,1.0
